# Stage 01 — Parsing

**Track A (Buse) · Stage 1 of 10**

| | |
|---|---|
| **Input** | `data/raw/*.pdf`, `data/corpus_manifest_B.jsonl` |
| **Output** | `data/interim/pages.jsonl`, one record per page with detected section |
| **Promotes to** | `src/research_assistant/ingestion/parse_B.py` |
| **Config** | `configs/ingestion_B.yaml` → `parse:` |

## What this stage decides

Parsing decides what information *can* exist downstream. A page number lost here is a
citation you cannot verify in stage 06, and a section heading missed here silently
downgrades section-aware chunking to fixed windows in stage 02.

The output is intentionally **pages, not chunks**. Keeping a page-level intermediate
means you can re-run every chunking experiment in stage 02 without re-parsing, which
is the slowest step in the pipeline.

## Design choices

| Choice | Picked | Alternatives | Why |
|---|---|---|---|
| Extractor | PyMuPDF | pdfplumber, GROBID, Nougat, marker | Fast, pure Python, no service to run, and it exposes font size and position, which is what makes heading detection possible. GROBID gives better section structure but is a Docker service Sude would have to host. |
| Unit of output | One record per page | Per document, per block | Page number is required for citation precision. Per-document loses it, per-block is more records than the chunker needs. |
| Section detection | Font-size and numbering heuristic | Regex on known headings, ML layout model | The heuristic covers most conference-format papers. Where it fails, chunking falls back to fixed windows, which is a degradation rather than a break. |
| References section | Dropped | Kept | Reference lists are dense keyword soup. They wreck BM25 by matching every query, and they never answer a question. |
| Failure handling | Flag, do not drop | Silently skip | A scanned page that yields 40 characters is a corpus problem you need to see, not a row to discard quietly. |

**If you want to change this:** the single highest-value swap is GROBID for
PyMuPDF, if section detection turns out to be the accuracy bottleneck in stage 02.
Record it in the ledger as `01 / layout-aware parsing` before you spend the day on it.

In [ ]:
from _nbsetup_B import REPO, DATA, load_cfg, resolve, ensure_dirs
import json, itertools
from pathlib import Path

cfg = load_cfg("ingestion")
cfg

In [ ]:
import pymupdf  # the `fitz` alias is deprecated
import re

HEADING_NUM = re.compile(r"^\s*(\d+(\.\d+)*)[\.\)]?\s+[A-Z]")

def page_blocks(page):
    # Returns (text, max_font_size, is_bold) per block, in reading order.
    out = []
    for b in page.get_text("dict")["blocks"]:
        if b.get("type") != 0:
            continue
        text, size, bold = [], 0.0, False
        for line in b["lines"]:
            for span in line["spans"]:
                text.append(span["text"])
                size = max(size, span["size"])
                bold = bold or "bold" in span["font"].lower()
        joined = " ".join(text).strip()
        if joined:
            out.append((joined, size, bold))
    return out

def looks_like_heading(text, size, bold, body_size):
    if len(text) > 90 or len(text.split()) > 12:
        return False
    if HEADING_NUM.match(text):
        return True
    return (size > body_size + 0.8) or (bold and size >= body_size)

In [ ]:
def parse_pdf(path, paper_id, drop_sections):
    doc = pymupdf.open(path)
    # Body font size = the most common span size across the document.
    sizes = {}
    for page in doc:
        for _, size, _ in page_blocks(page):
            sizes[round(size, 1)] = sizes.get(round(size, 1), 0) + 1
    body_size = max(sizes, key=sizes.get) if sizes else 10.0

    records, current_section = [], "FRONT_MATTER"
    for i, page in enumerate(doc, start=1):
        texts = []
        for text, size, bold in page_blocks(page):
            if looks_like_heading(text, size, bold, body_size):
                current_section = text
            texts.append(text)
        body = "\n".join(texts).strip()
        low = current_section.lower()
        records.append(dict(
            paper_id=paper_id,
            page=i,
            section=current_section,
            text=body,
            n_chars=len(body),
            dropped=any(d in low for d in drop_sections),
        ))
    doc.close()
    return records, body_size

In [ ]:
parse_cfg = cfg["parse"]
raw_dir = resolve(cfg["corpus"]["raw_dir"])
interim = resolve(cfg["corpus"]["interim_dir"])
ensure_dirs(interim)

manifest = {}
mpath = resolve(cfg["corpus"]["manifest"])
if mpath.exists():
    for line in mpath.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            manifest[r["filename"]] = r

all_pages, thin_pages = [], []
for pdf in sorted(raw_dir.glob("*.pdf")):
    pid = manifest.get(pdf.name, {}).get("paper_id", pdf.stem[:10])
    recs, body_size = parse_pdf(pdf, pid, parse_cfg["drop_sections"])
    for r in recs:
        if r["n_chars"] < parse_cfg["min_chars_per_page"] and not r["dropped"]:
            thin_pages.append((pdf.name, r["page"], r["n_chars"]))
    all_pages.extend(recs)
    print(f"{pdf.name:50s} pages={len(recs):3d} body_font={body_size}")

print(f"\ntotal pages: {len(all_pages)}")
print(f"pages below min_chars (inspect these, do not ignore): {len(thin_pages)}")
for t in thin_pages[:10]:
    print("  ", t)

### Check before moving on

Open three parsed pages by hand and read them. Automated checks catch empty output,
they do not catch two-column text interleaved into nonsense, which is the classic
PyMuPDF failure on conference formats.

In [ ]:
kept = [p for p in all_pages if not p["dropped"] and p["n_chars"] >= parse_cfg["min_chars_per_page"]]
out = interim / "pages.jsonl"
with out.open("w", encoding="utf-8") as f:
    for r in kept:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"wrote {len(kept)} pages to {out}  (dropped {len(all_pages) - len(kept)})")

# Eyeball the section detector: how many distinct sections per paper?
import collections
per_paper = collections.defaultdict(set)
for r in kept:
    per_paper[r["paper_id"]].add(r["section"])
counts = sorted((len(v), k) for k, v in per_paper.items())
print("\nfewest sections detected (these are where heading detection failed):")
for n, pid in counts[:5]:
    print(f"  {pid}: {n} sections")

## Exit checks

- [ ] Every paper in the manifest produced at least one page.
- [ ] Median sections per paper is between 5 and 15. Below 3 means heading detection
      failed and stage 02 will silently degrade to fixed windows.
- [ ] You read three random pages and the text is in reading order.
- [ ] Reference sections are gone.

## Promote to `src/`

Move `page_blocks`, `looks_like_heading` and `parse_pdf` into
`src/research_assistant/ingestion/parse_B.py`, with the heuristics' thresholds
read from config rather than hard-coded. The notebook keeps the exploration and
the failure gallery.